# 02 · Análisis exploratorio del consumo eléctrico de Brasil

Para cumplir el enunciado usamos [Brazil's energy consumption, lucasdasilvadantas](https://www.kaggle.com/datasets/lucasdasilvadantas/brazils-energy-consumption),
publicado a partir de datos de **EPE**. Es un Excel de registros mensuales por UF y segmento.
Su período y dimensiones se calculan abajo; no trasladamos cifras de la otra versión del repo.

Preguntas: ¿cómo se distribuyen consumo y consumidores? ¿Qué diferencias hay por mercado y sector?
¿Cómo varía el consumo mensual? ¿Qué estados concentran el consumo registrado?
Esta fuente contiene consumo y unidades consumidoras, no precio ni horarios baratos.
Orden obligatorio: exploración → limpieza → análisis simple → análisis complejo.

## Cómo ejecutar este notebook
Instalá las dependencias desde la raíz del proyecto con `python -m pip install -r requirements.txt`,
usando el mismo entorno Python que el kernel. Después reiniciá el kernel y ejecutá todas las celdas.
Necesitás conexión a internet para descargar el dataset público de Kaggle.
El archivo se lee en memoria y los resultados y gráficos quedan dentro del notebook.
En Kaggle, habilitá internet y vinculá el dataset correspondiente al publicar.
Una revisión del dataset puede cambiar los resultados; se imprime el hash del archivo descargado.

In [ ]:
from io import BytesIO
from urllib.request import urlopen
from zipfile import ZipFile
import hashlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

# Configuración visual y semilla para las muestras.
sns.set_theme(style="whitegrid", palette="colorblind")
pd.set_option("display.max_columns", 12)
SEMILLA = 42

# Descarga el dataset público de Kaggle en cada ejecución.
URL = "https://www.kaggle.com/api/v1/datasets/download/lucasdasilvadantas/brazils-energy-consumption"
with urlopen(URL, timeout=90) as respuesta:
    comprimido = respuesta.read()

with ZipFile(BytesIO(comprimido)) as zip_datos:
    contenido = zip_datos.read("Dados brutos.xlsx")

raw = pd.read_excel(BytesIO(contenido), engine="calamine")

print("Dataset:", "lucasdasilvadantas/brazils-energy-consumption")
print("Archivo:", "Dados brutos.xlsx", "| Filas y columnas:", raw.shape)
print("SHA-256:", hashlib.sha256(contenido).hexdigest())


# Ajusta y muestra los gráficos; después cierra las figuras abiertas.
def mostrar():
    plt.tight_layout()
    plt.show()
    plt.close("all")

## 1. Exploración de las características
| Columna | Significado |
|---|---|
| Data | Mes de referencia, entero AAAAMMDD |
| UF / Sistema | Estado brasileño / sistema eléctrico |
| TipoConsumidor | Cativo (regulado) / Livre (libre) |
| Setor Econômico - N1/N2/N3 | Clasificación jerárquica del sector |
| Tipo Tensão - N1/N2/N3 | Clasificación de tensión |
| Faixa de Consumo N1/N2 | Franja de consumo; sus etiquetas en kWh no cambian la unidad de Consumo |
| Consumo | Energía registrada en MWh |
| Consumidores | Cantidad de unidades consumidoras de ese segmento, no población |

Una fila es **un mes de un segmento**, no una factura ni un consumidor individual.
Varios segmentos comparten fecha; por eso las fechas repetidas son válidas.
Fuente institucional: [EPE](https://www.epe.gov.br/pt/publicacoes-dados-abertos/dados-abertos/dados-do-consumo-mensal-de-energia-eletrica).

In [ ]:
# Revisamos la estructura y los faltantes antes de preparar los datos.
display(raw.head())
raw.info()
display(raw[["Consumo", "Consumidores"]].describe().round(2))

display(pd.DataFrame({
    "nulos": raw.isna().sum(),
    "porcentaje": raw.isna().mean() * 100,
}).round(2))
print("Filas idénticas:", raw.duplicated().sum())

for columna in ["TipoConsumidor", "Sistema", "UF", "Setor Econômico - N1"]:
    display(raw[columna].value_counts().to_frame("registros"))

## 2. Limpieza y preprocesamiento
Conservamos `raw`. Renombramos el sector principal a `Classe`, como en la referencia.
Convertimos Data con formato explícito y ordenamos. Todas las dimensiones junto con la fecha
forman la clave de un segmento; si esa clave se repite, se debe investigar, no promediar en silencio.

Conservamos nulos y valores no positivos en `base`. Para relaciones consumo/consumidores usamos `df`
con ambas mediciones presentes. Para sumas mensuales usamos todo Consumo disponible en `base`,
aunque falte Consumidores, y mostramos cobertura. Un total observado con faltantes puede subestimar
el total verdadero; no rellenamos automáticamente con cero ni mediana.

In [ ]:
base = raw.copy().rename(columns={"Setor Econômico - N1": "Classe"})
dimensiones = [c for c in base.columns if c not in ["Data", "Consumo", "Consumidores"]]

for columna in dimensiones:
    base[columna] = base[columna].astype("string").str.strip()

# Validamos las fechas mensuales y la clave que identifica cada segmento.
base["Data"] = pd.to_datetime(base["Data"].astype(str), format="%Y%m%d", errors="raise")
assert base["Data"].dt.day.eq(1).all(), "Revisar frecuencia mensual"
assert not base.duplicated(["Data"] + dimensiones).any(), "Clave de segmento repetida"

base = base.sort_values("Data").set_index("Data")
df = base.dropna(subset=["Consumo", "Consumidores"]).copy()

# Registramos las exclusiones de df y los valores no positivos de base.
auditoria = pd.Series({
    "filas_originales": len(base),
    "filas_con_ambas_mediciones": len(df),
    "sin_alguna_medicion": len(base) - len(df),
    "consumo_negativo": int(base["Consumo"].lt(0).sum()),
    "consumo_cero": int(base["Consumo"].eq(0).sum()),
    "consumidores_no_positivos": int(base["Consumidores"].le(0).sum()),
    "meses": base.index.to_period("M").nunique(),
})
display(auditoria.to_frame("resultado"))
print("Período:", base.index.min().date(), "→", base.index.max().date())
display(base.loc[base["Consumo"] < 0, ["UF", "TipoConsumidor", "Consumo"]].head())

### Números negativos, ceros y la etiqueta TOTAL
Un logaritmo solo admite valores positivos. Creamos subconjuntos positivos para esas visualizaciones,
pero no borramos los valores no positivos de los totales. La fuente no prueba que cada negativo
sea inyección a la red; no atribuimos esa causa. Incluso hay negativos en registros Cativo.

`TOTAL` en un nivel inferior puede significar que no existe desglose. Verificamos en este archivo
que, para el mismo grupo padre, no coexistan ese subtotal y sus detalles en los niveles de sector/tensión.
La verificación evita sumar dos veces el mismo agregado en las dimensiones presentes.

In [ ]:
padres = ["UF", "Sistema", "TipoConsumidor", "Classe"]
tension = ["Tipo Tensão - N1", "Tipo Tensão - N2", "Tipo Tensão - N3"]
franjas = ["Faixa de Consumo N1", "Faixa de Consumo N2"]

# Para cada nivel, definimos las columnas que identifican al grupo padre.
chequeos = [
    ("Setor Econômico - N2", padres + tension + franjas),
    ("Setor Econômico - N3", padres + ["Setor Econômico - N2"] + tension + franjas),
    (
        "Tipo Tensão - N2",
        padres + ["Setor Econômico - N2", "Setor Econômico - N3", "Tipo Tensão - N1"]
        + franjas,
    ),
    (
        "Tipo Tensão - N3",
        padres + [
            "Setor Econômico - N2", "Setor Econômico - N3",
            "Tipo Tensão - N1", "Tipo Tensão - N2",
        ] + franjas,
    ),
]

# Detectamos si un mismo grupo contiene TOTAL junto con sus detalles.
revision = {}
for nivel, grupo in chequeos:
    grupos = base.reset_index().groupby(["Data"] + grupo)[nivel]
    conflicto = grupos.agg(lambda s: "TOTAL" in set(s) and len(set(s)) > 1)
    revision[nivel] = int(conflicto.sum())

display(pd.Series(revision).to_frame("grupos con TOTAL y detalle"))
assert not any(revision.values()), "Revisar jerarquía antes de sumar"

positivos = df.loc[df["Consumo"].gt(0) & df["Consumidores"].gt(0)].copy()
print("Registros completos y positivos para escala log:", len(positivos))

## 3. Análisis simple
### 3.1 Distribuciones individuales
Como en el repo de referencia, mostramos histograma y boxplot de consumo y consumidores.
La escala logarítmica permite ver valores de tamaños muy distintos. Media y mediana se calculan
sobre el subconjunto positivo de cada variable; su denominador se muestra en la tabla.

In [ ]:
fig, ejes = plt.subplots(2, 2, figsize=(12, 7))
resumen_distribuciones = {}

# Cada fila compara el histograma y el boxplot de una variable positiva.
for fila, columna, unidad in [
    (0, "Consumo", "MWh"),
    (1, "Consumidores", "unidades consumidoras"),
]:
    valores = base.loc[base[columna] > 0, columna]
    resumen_distribuciones[columna] = {
        "n": len(valores),
        "media": valores.mean(),
        "mediana": valores.median(),
    }

    sns.histplot(x=valores, bins=40, log_scale=True, ax=ejes[fila, 0])
    ejes[fila, 0].axvline(valores.mean(), color="red", ls="--", label="Media")
    ejes[fila, 0].axvline(valores.median(), color="green", ls=":", label="Mediana")
    ejes[fila, 0].set(
        title=columna + " · valores positivos",
        xlabel=unidad + " (escala log)", ylabel="Registros",
    )
    ejes[fila, 0].legend()

    sns.boxplot(x=valores, log_scale=True, ax=ejes[fila, 1])
    ejes[fila, 1].set(title="Boxplot de " + columna, xlabel=unidad + " (escala log)")

mostrar()
display(pd.DataFrame(resumen_distribuciones).T.round(2))

### 3.2 Distribuciones segmentadas y filtros
Comparamos Cativo/Livre y sectores usando registros positivos para la escala log.
El filtro sectorial Residencial/Industrial responde una comparación concreta.
La mediana por registro no es consumo medio por cliente: las filas son segmentos de tamaños diferentes.

In [ ]:
por_tipo = base.groupby("TipoConsumidor")["Consumo"].agg(["count", "mean", "median"])
display(por_tipo.round(2))

# El filtro sectorial usa registros positivos para graficar en escala log.
sectores = positivos.loc[positivos["Classe"].isin(["Residencial", "Industrial"])]
fig, ejes = plt.subplots(1, 2, figsize=(12, 4))

sns.boxplot(
    data=positivos, x="TipoConsumidor", y="Consumo", showfliers=False, ax=ejes[0],
)
ejes[0].set_yscale("log")
ejes[0].set(
    title="Consumo por tipo · registros positivos",
    xlabel="Mercado", ylabel="MWh (log)",
)

sns.histplot(
    data=sectores, x="Consumo", hue="Classe", bins=35, log_scale=True,
    stat="density", common_norm=False, element="step", ax=ejes[1],
)
ejes[1].set(
    title="Filtro: Residencial vs. Industrial",
    xlabel="Consumo positivo (MWh, log)", ylabel="Densidad en escala log",
)
mostrar()

### 3.3 Segmentación temporal por mes y año
Un boxplot por mes compara las distribuciones de registros positivos, no los totales nacionales.
Para resumir los años mostramos media, mediana y banda P25–P75 entre registros.
Esa banda es dispersión del dataset; no es un intervalo de confianza.

In [ ]:
positivos["mes"] = positivos.index.month
positivos["anio"] = positivos.index.year
anual_registros = positivos.groupby("anio")["Consumo"].describe()

fig, ejes = plt.subplots(1, 2, figsize=(13, 4))
sns.boxplot(data=positivos, x="mes", y="Consumo", showfliers=False, ax=ejes[0])
ejes[0].set_yscale("log")
ejes[0].set(
    title="Consumo por mes · registros positivos",
    xlabel="Mes", ylabel="MWh por registro (log)",
)

# La banda muestra la dispersión entre los percentiles 25 y 75 de cada año.
ejes[1].fill_between(
    anual_registros.index, anual_registros["25%"], anual_registros["75%"],
    alpha=0.2, label="P25–P75",
)
ejes[1].plot(anual_registros.index, anual_registros["mean"], label="Media")
ejes[1].plot(anual_registros.index, anual_registros["50%"], label="Mediana")
ejes[1].set(
    title="Resumen anual entre registros positivos",
    xlabel="Año", ylabel="MWh por registro",
)
ejes[1].legend()
mostrar()

display(anual_registros[["count", "mean", "50%", "25%", "75%"]].round(2))

### 3.4 Correlación y recta descriptiva
Pearson sobre registros completos mide asociación lineal, no causalidad.
La recta ajustada es una visualización descriptiva, no un modelo validado para predecir.
Usamos todos los registros completos para correlación y hasta 1.000 para la dispersión/recta.
No afirmamos que cantidad de consumidores sea el principal motor del consumo a partir de esto.

In [ ]:
# La correlación usa todos los registros con ambas mediciones.
correlacion = df[["Consumidores", "Consumo"]].corr()
display(correlacion.round(3))

muestra_completa = df.sample(n=min(1000, len(df)), random_state=SEMILLA)
fig, ejes = plt.subplots(1, 2, figsize=(12, 4))
sns.heatmap(
    correlacion, annot=True, vmin=-1, vmax=1, center=0, cmap="vlag", ax=ejes[0],
)
ejes[0].set_title("Pearson · registros completos (sin unidad)")

# La dispersión y la recta descriptiva se muestran sobre la muestra fija.
sns.regplot(
    data=muestra_completa, x="Consumidores", y="Consumo", ci=None,
    scatter_kws={"s": 10, "alpha": 0.25}, line_kws={"color": "red"}, ax=ejes[1],
)
ejes[1].set(
    title="Recta descriptiva · muestra de registros",
    xlabel="Unidades consumidoras", ylabel="Consumo (MWh)",
)
mostrar()

### 3.5 Pairplot
Para mejorar legibilidad transformamos dos variables positivas a log10.
Un valor 3 en el eje log10 representa 1.000 en la unidad original.
No sustituimos con logaritmos las variables de la correlación anterior.
La muestra fija es solo para visualización; se etiqueta por tipo de consumidor.

In [ ]:
# El log10 se calcula solo sobre la muestra de registros positivos.
muestra = positivos.sample(n=min(1000, len(positivos)), random_state=SEMILLA).copy()
muestra["log10_consumo_MWh"] = np.log10(muestra["Consumo"])
muestra["log10_consumidores"] = np.log10(muestra["Consumidores"])

g = sns.pairplot(
    muestra, vars=["log10_consumo_MWh", "log10_consumidores"],
    hue="TipoConsumidor", diag_kind="hist", plot_kws={"alpha": 0.3, "s": 10},
)
g.fig.suptitle("Brasil · Pairplot en log10 de 1.000 registros positivos", y=1.02)
mostrar()

## 4. Análisis complejo
### 4.1 Serie mensual y media móvil de 12 meses
Sumamos MWh de segmentos distintos del mismo mes. Energía se suma; una media entre segmentos
respondería otra pregunta. `sum(min_count=1)` evita convertir un mes sin datos en cero.
La cobertura mide registros con dato sobre filas declaradas del mes, no segmentos que nunca aparecieron.
La media móvil es retrospectiva y usa 12 meses completos, sin ventana centrada ni datos futuros.

In [ ]:
# min_count=1 conserva como faltante un mes sin mediciones disponibles.
mensual = base[["Consumo", "Consumidores"]].resample("MS").sum(min_count=1)
mensual["registros"] = base["Consumo"].resample("MS").size()
mensual["consumo_presente"] = base["Consumo"].resample("MS").count()
mensual["cobertura_pct"] = mensual["consumo_presente"] / mensual["registros"] * 100

# La ventana requiere doce meses y usa únicamente el mes actual y los anteriores.
mensual["media_movil_12"] = mensual["Consumo"].rolling(12, min_periods=12).mean()
display(mensual.head(12).round(2))

fig, ejes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
ejes[0].plot(
    mensual.index, mensual["Consumo"] / 1e6,
    alpha=0.5, label="Total observado mensual",
)
ejes[0].plot(
    mensual.index, mensual["media_movil_12"] / 1e6,
    lw=2, label="Media móvil de 12 meses",
)
ejes[0].set(
    title="Consumo registrado en Brasil y nivel suavizado",
    ylabel="Millones de MWh/mes",
)
ejes[0].legend()

ejes[1].plot(mensual.index, mensual["Consumidores"] / 1e6)
ejes[1].set(
    title="Unidades consumidoras por mes · suma de segmentos",
    xlabel="Mes", ylabel="Millones de unidades",
)
mostrar()

### 4.2 Patrón por mes: comparar años, no copiar una estación esperada
El heatmap año/mes permite comprobar si los meses altos se repiten.
Los totales mensuales también cambian por crecimiento y composición, por lo que un pico
promedio no demuestra que el clima lo cause. No asumimos el patrón diciembre–marzo de otra versión.

In [ ]:
# Organizamos los totales mensuales en una matriz de año por mes.
calendario = mensual.copy()
calendario["anio"] = calendario.index.year
calendario["mes"] = calendario.index.month
matriz = calendario.pivot(index="anio", columns="mes", values="Consumo")

plt.figure(figsize=(11, 5))
sns.heatmap(matriz / 1e6, annot=True, fmt=".1f", cmap="YlOrRd")
plt.title("Consumo mensual observado (millones de MWh) · Brasil")
plt.xlabel("Mes")
plt.ylabel("Año")
mostrar()

patron_mes = calendario.groupby("mes")["Consumo"].agg(["count", "mean", "median"])
display(patron_mes.round(2))

### 4.3 Estados y sectores
Ahora usamos UF, disponible en el dataset del enunciado. Sumamos energía de todo el período
observado y mostramos sus cinco estados principales. El número de filas no indica consumo.
Para consumidores mostramos un mes concreto: sumarlos entre años no cuenta personas únicas.

In [ ]:
# Ordenamos las UF por el consumo acumulado de todo el período.
por_uf = base.groupby("UF")["Consumo"].sum(min_count=1).sort_values(ascending=False)
top_uf = por_uf.head(5)
por_sector = base.pivot_table(index="UF", columns="Classe", values="Consumo", aggfunc="sum")

fig, ejes = plt.subplots(1, 2, figsize=(13, 5))
(top_uf / 1e6).plot.bar(ax=ejes[0])
ejes[0].set(
    title="Cinco UF con más consumo registrado",
    xlabel="Estado", ylabel="Millones de MWh · período completo",
)

(por_sector.loc[top_uf.index] / 1e6).plot.bar(stacked=True, ax=ejes[1])
ejes[1].set(
    title="Consumo por UF y sector",
    xlabel="Estado", ylabel="Millones de MWh · período completo",
)
ejes[1].legend(title="Sector", bbox_to_anchor=(1.02, 1), fontsize=7)
mostrar()
display(top_uf.to_frame("MWh observados"))

# Para unidades consumidoras usamos únicamente el último mes disponible.
ultimo_mes = mensual.index.max()
display(
    base.loc[base.index == ultimo_mes].groupby("UF")["Consumidores"]
    .sum(min_count=1)
    .nlargest(5)
    .to_frame("unidades en último mes")
)

### 4.4 Cativo vs. Livre: total e intensidad entre grupos comparables
Agrupamos por mes y tipo, sumando consumo/consumidores de registros donde ambas mediciones existen.
El cociente es MWh por unidad consumidora de ese mes, no consumo per cápita ni precio.
Un mercado puede tener pocos consumidores con consumo elevado; hay que comparar tanto totales como ratios.

In [ ]:
# Agrupamos registros completos para que numerador y denominador sean comparables.
por_mercado = df.groupby([pd.Grouper(freq="MS"), "TipoConsumidor"])[
    ["Consumo", "Consumidores"]
].sum(min_count=1)
por_mercado["MWh_por_unidad"] = por_mercado["Consumo"] / por_mercado["Consumidores"].where(
    por_mercado["Consumidores"] > 0
)

fig, ejes = plt.subplots(1, 2, figsize=(12, 4))
(por_mercado["Consumo"].unstack() / 1e6).plot(ax=ejes[0])
ejes[0].set(
    title="Consumo de registros completos por mercado",
    xlabel="Mes", ylabel="Millones de MWh/mes",
)

por_mercado["MWh_por_unidad"].unstack().plot(ax=ejes[1])
ejes[1].set(
    title="Intensidad observada por mercado",
    xlabel="Mes", ylabel="MWh/unidad consumidora/mes",
)
mostrar()

display(por_mercado.tail(6).round(2))

## Conclusiones y límites
Los números salen del dataset ejecutado. Los faltantes hacen que los totales sean observados,
no estimaciones completas; los no positivos requieren contexto adicional de la fuente.
No inferimos causalidad, consumo individual, población, tarifas ni energía barata a partir de este archivo.

In [ ]:
# Reunimos los resultados calculados para redactar las conclusiones.
anual_total = mensual["Consumo"].groupby(mensual.index.year).sum(min_count=1)

resumen = {
    "filas_originales": len(base),
    "filas_completas": len(df),
    "sin_alguna_medicion": len(base) - len(df),
    "meses": len(mensual),
    "inicio": str(base.index.min().date()),
    "fin": str(base.index.max().date()),
    "nulos_consumo": int(base["Consumo"].isna().sum()),
    "nulos_consumidores": int(base["Consumidores"].isna().sum()),
    "consumo_negativo": int(base["Consumo"].lt(0).sum()),
    "consumo_cero": int(base["Consumo"].eq(0).sum()),
    "pearson_consumidores_consumo": float(correlacion.loc["Consumidores", "Consumo"]),
    "media_consumo_positivo": float(resumen_distribuciones["Consumo"]["media"]),
    "mediana_consumo_positivo": float(resumen_distribuciones["Consumo"]["mediana"]),
    "cobertura_min_pct": float(mensual["cobertura_pct"].min()),
    "top_uf_mwh": top_uf.to_dict(),
    "consumo_anual_mwh": anual_total.to_dict(),
    "mes_media_mayor": int(patron_mes["mean"].idxmax()),
    "mes_media_menor": int(patron_mes["mean"].idxmin()),
}

display(Markdown(f"""
- Se inspeccionaron **{len(base):,} registros** de **{len(mensual)} meses**, entre {resumen['inicio']} y {resumen['fin']}.
- **{len(base) - len(df):,} registros** no tienen ambas mediciones; solo se excluyen de las comparaciones que requieren las dos.
- Hay **{resumen['consumo_negativo']} consumos negativos** y **{resumen['consumo_cero']} ceros**: se contabilizan, sin asignarles una causa no verificada.
- Entre consumos positivos, media **{resumen['media_consumo_positivo']:,.2f} MWh** y mediana **{resumen['mediana_consumo_positivo']:,.2f} MWh**, consistente con cola derecha.
- Consumo y unidades consumidoras tienen Pearson **{resumen['pearson_consumidores_consumo']:.3f}** en registros completos: asociación descriptiva.
- **{top_uf.index[0]}** lidera los totales registrados con **{top_uf.iloc[0] / 1e6:,.2f} millones de MWh** en todo el período, sin ajuste por población.
- En la media por mes calendario, el mes **{resumen['mes_media_mayor']}** es el mayor y **{resumen['mes_media_menor']}** el menor. Debe leerse junto al mapa de cada año.
"""))

In [ ]:

# PROXY DE ENERGÍA POTENCIALMENTE BARATA
# Puntaje heurístico de 0 a 10

proxy = positivos.copy()

# Consumo medio por unidad consumidora
proxy["consumo_medio_mwh"] = (
    proxy["Consumo"] / proxy["Consumidores"]
)

# Definimos consumo alto como el 25% superior
umbral_alto = proxy["consumo_medio_mwh"].quantile(0.75)

# 1. Indicadores (flags)
proxy["is_livre"] = (
    proxy["TipoConsumidor"] == "Livre"
).astype(int)

proxy["is_alta_tensao"] = (
    proxy["Tipo Tensão - N1"]
    .str.contains("Alta", case=False, na=False)
).astype(int)

proxy["is_sin"] = (
    ~proxy["Sistema"].str.contains(
        "Isolad", case=False, na=False
    )
).astype(int)

proxy["is_industrial"] = (
    proxy["Classe"] == "Industrial"
).astype(int)

proxy["is_consumo_alto"] = (
    proxy["consumo_medio_mwh"] >= umbral_alto
).astype(int)

proxy["is_baixa_tensao"] = (
    proxy["Tipo Tensão - N1"]
    .str.contains("Baixa", case=False, na=False)
).astype(int)

# 2. Puntaje de condiciones potencialmente favorables
proxy["proxy_barato_score"] = (
    proxy["is_livre"] * 3
    + proxy["is_alta_tensao"] * 2
    + proxy["is_sin"] * 2
    + proxy["is_industrial"]
    + proxy["is_consumo_alto"]
    + proxy["is_consumo_alto"] * proxy["is_livre"]
    - (1 - proxy["is_sin"]) * 3
    - proxy["is_baixa_tensao"] * 2
).clip(lower=0, upper=10)

# 3. Comparación por estado durante 2023
datos_2023 = proxy.loc[
    (proxy.index.year >=1990)
    & (proxy["Classe"] == "Industrial")
]

ranking = datos_2023.groupby("UF").agg(
    puntaje_medio=("proxy_barato_score", "mean"),
    puntaje_mediano=("proxy_barato_score", "median"),
    registros=("proxy_barato_score", "size"),
    pct_favorables=(
        "proxy_barato_score",
        lambda x: (x >= 7).mean() * 100
    )
)

ranking = ranking.sort_values(
    "pct_favorables", ascending=False
)

display(ranking.round(2).head(10))

# 4. Visualización
plt.figure(figsize=(11, 5))

sns.barplot(
    data=ranking.head(10).reset_index(),
    x="UF",
    y="pct_favorables",
    color="steelblue"
)

plt.title("Estados con mayor proporción de perfiles favorables (2023)")
plt.xlabel("Estado de Brasil")
plt.ylabel("Perfiles con puntaje >= 7 (%)")
plt.ylim(0, 100)
plt.tight_layout()
plt.show()



### Conclusión: Proxy de energía potencialmente barata

El análisis permitió identificar los estados de Brasil que presentan condiciones potencialmente favorables para acceder a energía eléctrica a menor costo durante 2023, considerando factores como el mercado libre, la alta tensión, la conexión al sistema interconectado y el nivel de consumo industrial.

Los resultados muestran que **Mato Grosso (MT) presenta la mayor proporción de perfiles favorables, con un 56,07 %**, seguido de **Río de Janeiro (RJ) y Santa Catarina (SC), ambos con un 55,56 %**. Los demás estados del ranking presentan un 50 % de registros con puntajes iguales o superiores a 7.

Esto indica que Mato Grosso reúne la mayor proporción de registros industriales con características consideradas favorables según nuestro modelo. Sin embargo, **no significa que tenga la electricidad más barata de Brasil**, ya que el análisis no utiliza tarifas eléctricas reales, sino un puntaje estimado a partir de las condiciones de consumo y suministro.

**En conclusión**, el proxy permite identificar posibles ventajas energéticas para el sector industrial y comparar los estados bajo un mismo criterio. No obstante, los resultados deben interpretarse como una aproximación y no como una confirmación de menores precios de electricidad.
